In [2]:
import numpy as np
import pandas as pd
import h5py
import pysam
import os
from modisco.visualization import viz_sequence
from modisco import util
from matplotlib import pyplot as plt
import pybedtools
from bs4 import BeautifulSoup

pd.options.display.max_rows = 500
pd.options.display.max_columns = 500

In [3]:
# mode="print"
# modisco_path = '/mnt/lab_data2/anusri/chrombpnet/results/print/ATAC_PE/ENCSR158XTU/chrombpnet_compare/ENCSR158XTU/auxiliary/interpret_subsample/old_modisco_results_profile_scores.h5'
# ppm_dir = 'pwms/'

# htmld="/mnt/lab_data2/anusri/chrombpnet/results/print/ATAC_PE/ENCSR158XTU/chrombpnet_compare/ENCSR158XTU/evaluation/modisco_profile/motifs.html"
# tomtom = pd.read_html(htmld)

# mode="gm12878_print"
# modisco_path = '/mnt/lab_data2/anusri/chrombpnet/results/print/ATAC_PE/GM12878/ATAC_PE_05.08.2025_print_with_bias_bigwig/print_model/interpret/profile/old_modisco_results_profile_scores.h5'
# ppm_dir = 'pwms/'

# htmld="/mnt/lab_data2/anusri/chrombpnet/results/print/ATAC_PE/GM12878/ATAC_PE_05.08.2025_print_with_bias_bigwig/print_model/interpret/profile/motifs.html"
# tomtom = pd.read_html(htmld)

mode="kero_print"
modisco_path = '/mnt/lab_data2/anusri/chrombpnet/results/print/ATAC_PE/ENCSR158XTU/ATAC_PE_05.09.2025_print_with_bias_bigwig/print_model/interpret_new/profile/old_modisco_profile.h5'
ppm_dir = 'pwms/'

htmld="/mnt/lab_data2/anusri/chrombpnet/results/print/ATAC_PE/ENCSR158XTU/ATAC_PE_05.09.2025_print_with_bias_bigwig/print_model/interpret_new/profile/motifs.html"
tomtom = pd.read_html(htmld)

background=[0.25, 0.25, 0.25, 0.25]

In [4]:
def trim_motif_new(cwm, motif, trim_threshold=0.20):
    """
    Given the PFM and motif (both L x 4 arrays) (the motif could be the
    PFM itself), trims `motif` by cutting off flanks of low information
    content in `pfm`. `min_ic` is the minimum required information
    content. If specified this trimmed motif will be extended on either
    side by `pad` bases.
    If no base passes the `min_ic` threshold, then no trimming is done.
    """
    
    score = np.sum(np.abs(cwm), axis=1)
    trim_thresh = np.max(score) * trim_threshold  # Cut off anything less than 30% of max score
    pass_inds = np.where(score >= trim_thresh)[0]
    trimmed = motif[np.min(pass_inds): np.max(pass_inds) + 1]
 
    if not trimmed.size:
        return motif
    
    return trimmed

def import_tfmodisco_motifs(tfm_results_path, trim=True, only_pos=True):
    """
    Imports the PFMs to into a dictionary, mapping `(x, y)` to the PFM,
    where `x` is the metacluster index and `y` is the pattern index.
    Arguments:
        `tfm_results_path`: path to HDF5 containing TF-MoDISco results
        `out_dir`: where to save motifs
        `trim`: if True, trim the motif flanks based on information content
        `only_pos`: if True, only return motifs with positive contributions
    Returns the dictionary of PFMs.
    """ 
    pfms = {}
    with h5py.File(tfm_results_path, "r") as f:
        metaclusters = f["metacluster_idx_to_submetacluster_results"]
        num_metaclusters = len(metaclusters.keys())
        for metacluster_i, metacluster_key in enumerate(metaclusters.keys()):
            metacluster = metaclusters[metacluster_key]
            if "patterns" not in metacluster["seqlets_to_patterns_result"]:
                continue
            patterns = metacluster["seqlets_to_patterns_result"]["patterns"]
            num_patterns = len(patterns["all_pattern_names"][:])
            for pattern_i, pattern_name in enumerate(patterns["all_pattern_names"][:]):
#                pattern_name = pattern_name.decode()
                pattern_name = pattern_name

                pattern = patterns[pattern_name]
                pfm = pattern["sequence"]["fwd"][:]
                cwm = pattern["task0_contrib_scores"]["fwd"][:]
                
                # Check that the contribution scores are overall positive
                if only_pos and np.sum(cwm) < 0:
                    continue
                    
                if trim:
                    pfm = trim_motif_new(cwm, cwm)
                else:
                    pfm = cwm
                    
                pfms["%d_%d" % (metacluster_i,pattern_i)] = pfm
    return pfms

In [5]:
pfms = import_tfmodisco_motifs(modisco_path, trim=False)

In [6]:
for key in pfms:
    f = open(os.path.join(ppm_dir,mode+"_"+key+".pfm"),"w")
    #print(pfms[key])
    np.savetxt(f, pfms[key], fmt='%f')
    f.close()
    

In [7]:
tomtom[0]

,pattern,num_seqlets,modisco_cwm_fwd,modisco_cwm_rev,match0,qval0,match0_logo,match1,qval1,match1_logo,match2,qval2,match2_logo
0,pos_patterns.pattern_0,10413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,pos_patterns.pattern_1,7966,NaN,NaN,FOSB_HUMAN.H11MO.0.A,1.020140e-04,NaN,FOSB_MOUSE.H11MO.0.A,1.020140e-04,NaN,JUN_HUMAN.H11MO.0.A,1.020140e-04,NaN
2,pos_patterns.pattern_2,2410,NaN,NaN,CTCF_MA0139.1,3.366060e-14,NaN,CTCF_HUMAN.H11MO.0.A,2.421640e-11,NaN,CTCF_MOUSE.H11MO.0.A,1.834120e-09,NaN
3,pos_patterns.pattern_3,1571,NaN,NaN,P73_HUMAN.H11MO.0.A,2.794620e-10,NaN,P73_MOUSE.H11MO.0.B,2.794620e-10,NaN,P63_HUMAN.H11MO.0.A,2.831130e-09,NaN
4,pos_patterns.pattern_4,681,NaN,NaN,KLF4_HUMAN.H11MO.0.A,4.082750e-06,NaN,KLF4_MOUSE.H11MO.0.A,1.793100e-05,NaN,KLF1_MOUSE.H11MO.0.A,1.829890e-05,NaN
5,pos_patterns.pattern_5,504,NaN,NaN,GRHL2_MA1105.1,4.192730e-05,NaN,GRHL2_MOUSE.H11MO.0.A,4.755160e-05,NaN,GRHL2_HUMAN.H11MO.0.A,4.895840e-05,NaN
6,pos_patterns.pattern_6,423,NaN,NaN,CEBPA_MA0102.3,5.820970e-08,NaN,CEBPB_HUMAN.H11MO.0.A,5.062910e-07,NaN,CEBPB_MOUSE.H11MO.0.A,6.118130e-06,NaN
7,pos_patterns.pattern_7,387,NaN,NaN,TFAP2A_MA0003.3,6.932970e-06,NaN,TFAP2A_TFAP_2,6.932970e-06,NaN,Tcfap2a.mouse_TFAP_2,1.153050e-05,NaN
8,pos_patterns.pattern_8,277,NaN,NaN,TEAD3_MA0808.1,1.118060e-02,NaN,TEAD3_TEA_2,1.118060e-02,NaN,FOSL2_MA0478.1,1.118060e-02,NaN
9,pos_patterns.pattern_9,263,NaN,NaN,TEAD3_MA0808.1,3.336740e-03,NaN,TEAD3_TEA_2,3.336740e-03,NaN,TEAD2_MOUSE.H11MO.0.C,3.759640e-02,NaN


In [8]:

tomtom[0]["pattern"] = tomtom[0]["pattern"].str.replace("pos_patterns.pattern","0").str.replace("neg_patterns.pattern","1")


/users/anusri/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: FutureWarning: The default value of regex will change from True to False in a future version.
  """Entry point for launching an IPython kernel.


In [9]:
tomtom[0]

,pattern,num_seqlets,modisco_cwm_fwd,modisco_cwm_rev,match0,qval0,match0_logo,match1,qval1,match1_logo,match2,qval2,match2_logo
0,0_0,10413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0_1,7966,NaN,NaN,FOSB_HUMAN.H11MO.0.A,1.020140e-04,NaN,FOSB_MOUSE.H11MO.0.A,1.020140e-04,NaN,JUN_HUMAN.H11MO.0.A,1.020140e-04,NaN
2,0_2,2410,NaN,NaN,CTCF_MA0139.1,3.366060e-14,NaN,CTCF_HUMAN.H11MO.0.A,2.421640e-11,NaN,CTCF_MOUSE.H11MO.0.A,1.834120e-09,NaN
3,0_3,1571,NaN,NaN,P73_HUMAN.H11MO.0.A,2.794620e-10,NaN,P73_MOUSE.H11MO.0.B,2.794620e-10,NaN,P63_HUMAN.H11MO.0.A,2.831130e-09,NaN
4,0_4,681,NaN,NaN,KLF4_HUMAN.H11MO.0.A,4.082750e-06,NaN,KLF4_MOUSE.H11MO.0.A,1.793100e-05,NaN,KLF1_MOUSE.H11MO.0.A,1.829890e-05,NaN
5,0_5,504,NaN,NaN,GRHL2_MA1105.1,4.192730e-05,NaN,GRHL2_MOUSE.H11MO.0.A,4.755160e-05,NaN,GRHL2_HUMAN.H11MO.0.A,4.895840e-05,NaN
6,0_6,423,NaN,NaN,CEBPA_MA0102.3,5.820970e-08,NaN,CEBPB_HUMAN.H11MO.0.A,5.062910e-07,NaN,CEBPB_MOUSE.H11MO.0.A,6.118130e-06,NaN
7,0_7,387,NaN,NaN,TFAP2A_MA0003.3,6.932970e-06,NaN,TFAP2A_TFAP_2,6.932970e-06,NaN,Tcfap2a.mouse_TFAP_2,1.153050e-05,NaN
8,0_8,277,NaN,NaN,TEAD3_MA0808.1,1.118060e-02,NaN,TEAD3_TEA_2,1.118060e-02,NaN,FOSL2_MA0478.1,1.118060e-02,NaN
9,0_9,263,NaN,NaN,TEAD3_MA0808.1,3.336740e-03,NaN,TEAD3_TEA_2,3.336740e-03,NaN,TEAD2_MOUSE.H11MO.0.C,3.759640e-02,NaN


In [10]:
tomtom[0][["pattern","num_seqlets"]].to_csv(os.path.join(ppm_dir,mode+"_counts.csv"),sep=",",index=False, header=False)